In [ ]:
# =============================================================================
# CELL 1 — Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_metadata_snapshot as ncbi_metadata_snapshot_module
import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_metadata_snapshot_module = importlib.reload(ncbi_metadata_snapshot_module)
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_ncbi_protein_metadata_snapshot = (
    ncbi_metadata_snapshot_module.resolve_ncbi_protein_metadata_snapshot
)

from src.pago_pipeline.storage import read_json_file, sha256_of_file

In [ ]:
# =============================================================================
# CELL 2 — Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your NCBI configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# =============================================================================
# CELL 3 — Define metadata snapshot configuration
# =============================================================================

# Notebook 02 expects notebook 01 to have already materialized the upstream
# consolidated XML snapshot under the latest/ directory below.
XML_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "01-raw" / "protein_xml_snapshots"
)
METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT / "data" / "02-intermediate" / "protein_metadata_csv"
)
METADATA_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create

# Optional: point this at an older metadata manifest to compare schema drift.
REFERENCE_METADATA_MANIFEST_FILE_PATH = None

print(f"XML snapshot root directory: {XML_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Metadata snapshot root directory: {METADATA_SNAPSHOT_ROOT_DIRECTORY}")
print(f"Metadata snapshot mode: {METADATA_SNAPSHOT_MODE}")
print(f"Reference manifest path: {REFERENCE_METADATA_MANIFEST_FILE_PATH}")

In [ ]:
# =============================================================================
# CELL 4 — Resolve active metadata snapshot
# =============================================================================

metadata_snapshot_payload = resolve_ncbi_protein_metadata_snapshot(
    snapshot_mode=METADATA_SNAPSHOT_MODE,
    snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    source_xml_snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
    reference_metadata_manifest_file_path=REFERENCE_METADATA_MANIFEST_FILE_PATH,
)

metadata_snapshot_directory = metadata_snapshot_payload["snapshot_directory"]
metadata_snapshot_manifest = metadata_snapshot_payload["manifest"]
metadata_snapshot_manifest_file_path = metadata_snapshot_payload["manifest_file_path"]
metadata_csv_file_path = metadata_snapshot_payload["csv_file_path"]
metadata_qc_report_file_path = metadata_snapshot_payload["qc_report_file_path"]

metadata_csv_file_sha256 = sha256_of_file(input_file_path=metadata_csv_file_path)
metadata_manifest_file_sha256 = sha256_of_file(
    input_file_path=metadata_snapshot_manifest_file_path,
)
metadata_qc_report_file_sha256 = sha256_of_file(
    input_file_path=metadata_qc_report_file_path,
)
metadata_qc_report_payload = read_json_file(
    input_file_path=metadata_qc_report_file_path,
)

source_xml_snapshot_relative_path = metadata_snapshot_manifest.get(
    "source_xml_snapshot_relative_path"
)
if not isinstance(source_xml_snapshot_relative_path, str) or not source_xml_snapshot_relative_path:
    raise RuntimeError(
        "Metadata snapshot manifest is missing source_xml_snapshot_relative_path."
    )

source_xml_file_name = metadata_snapshot_manifest.get("source_xml_file_name")
if not isinstance(source_xml_file_name, str) or not source_xml_file_name:
    raise RuntimeError("Metadata snapshot manifest is missing source_xml_file_name.")

source_xml_snapshot_directory = (
    XML_SNAPSHOT_ROOT_DIRECTORY / source_xml_snapshot_relative_path
)
source_xml_file_path = source_xml_snapshot_directory / source_xml_file_name

print("Metadata snapshot resolved successfully.")
print(f"Snapshot directory: {metadata_snapshot_directory}")
print(f"Manifest path: {metadata_snapshot_manifest_file_path}")
print(f"CSV path: {metadata_csv_file_path}")
print(f"QC report path: {metadata_qc_report_file_path}")

In [ ]:
# =============================================================================
# CELL 5 — Print metadata snapshot summary
# =============================================================================

print("Metadata snapshot is ready.")
print(f"Snapshot created at UTC: {metadata_snapshot_manifest['snapshot_created_at_utc']}")
print(f"Row count: {metadata_snapshot_manifest['row_count']}")
print(f"Column count: {metadata_snapshot_manifest['column_count']}")
print(
    "Source XML snapshot relative path: "
    f"{metadata_snapshot_manifest['source_xml_snapshot_relative_path']}"
)

In [ ]:
# =============================================================================
# CELL 6 — Print metadata artifact summary
# =============================================================================

print("Frozen metadata snapshot artifacts:")
print(f"Snapshot directory: {metadata_snapshot_directory}")
print(f"CSV file path: {metadata_csv_file_path}")
print(f"CSV SHA-256: {metadata_csv_file_sha256}")
print(f"Manifest file path: {metadata_snapshot_manifest_file_path}")
print(f"Manifest SHA-256: {metadata_manifest_file_sha256}")
print(f"QC report path: {metadata_qc_report_file_path}")
print(f"QC report SHA-256: {metadata_qc_report_file_sha256}")
print(f"XML source path: {source_xml_file_path}")
print(f"XML source SHA-256: {metadata_snapshot_manifest['source_xml_file_sha256']}")

In [ ]:
# =============================================================================
# CELL 7 — Print schema summary
# =============================================================================

first_twenty_columns = metadata_snapshot_manifest["columns"][:20]
first_ten_feature_keys = metadata_snapshot_manifest["observed_feature_keys"][:10]

print("Flattened metadata schema summary:")
print(f"Maximum taxonomy depth observed: {metadata_snapshot_manifest['max_taxonomy_depth']}")
print(
    "Observed feature key count: "
    f"{len(metadata_snapshot_manifest['observed_feature_keys'])}"
)
print(f"First 10 feature keys: {first_ten_feature_keys}")
print(f"First 20 CSV columns: {first_twenty_columns}")

In [ ]:
# =============================================================================
# CELL 8 — Print provenance summary
# =============================================================================

print("Snapshot provenance summary:")
print(f"Metadata snapshot directory: {metadata_snapshot_directory}")
print(
    "Metadata snapshot relative path: "
    f"{metadata_snapshot_manifest['immutable_snapshot_relative_path']}"
)
print(f"Source XML snapshot directory: {source_xml_snapshot_directory}")
print(
    "Source XML snapshot relative path: "
    f"{metadata_snapshot_manifest['source_xml_snapshot_relative_path']}"
)
print(
    "Source XML snapshot manifest SHA-256: "
    f"{metadata_snapshot_manifest['source_xml_snapshot_manifest_sha256']}"
)
print(
    f"Source XML retrieved_at_utc: {metadata_snapshot_manifest['source_xml_retrieved_at_utc']}"
)
print(f"Search query: {metadata_snapshot_manifest['search_query']}")

In [ ]:
# =============================================================================
# CELL 9 — Expose downstream variables
# =============================================================================

metadata_csv_output_directory = metadata_snapshot_directory
metadata_csv_output_file_path = metadata_csv_file_path
metadata_csv_manifest_file_path = metadata_snapshot_manifest_file_path
metadata_snapshot_manifest_payload = metadata_snapshot_manifest
metadata_qc_output_directory = metadata_snapshot_directory
metadata_qc_result = metadata_snapshot_manifest["qc_summary"]
source_xml_snapshot_payload = metadata_snapshot_manifest

print("Variables exposed for downstream notebooks:")
print("- metadata_csv_output_directory")
print("- metadata_csv_output_file_path")
print("- metadata_csv_manifest_file_path")
print("- metadata_snapshot_manifest_payload")
print("- metadata_qc_output_directory")
print("- metadata_qc_report_file_path")
print("- metadata_qc_result")
print("- metadata_qc_report_payload")
print("- source_xml_snapshot_payload")
print("- source_xml_snapshot_directory")
print("- source_xml_file_path")

In [ ]:
# =============================================================================
# CELL 10 — Inspect exported CSV preview
# =============================================================================

preview_row_limit = 3
preview_column_limit = 20

preview_dataframe = pd.read_csv(
    metadata_csv_output_file_path,
    nrows=preview_row_limit,
)

print(f"CSV column count: {len(preview_dataframe.columns)}")
print(f"First 20 column names: {list(preview_dataframe.columns[:preview_column_limit])}")

preview_dataframe.iloc[:, :preview_column_limit]

In [ ]:
# =============================================================================
# CELL 11 — Print frozen QC summary
# =============================================================================

metadata_qc_summary = metadata_snapshot_manifest["qc_summary"]

print("Frozen QC summary:")
for check_name, check_value in metadata_qc_report_payload["checks"].items():
    print(f"- {check_name}: {check_value}")

print(
    f"Empty protein_uid count: {metadata_qc_summary['empty_protein_uid_count']}"
)
print(
    "Duplicate protein_uid count: "
    f"{metadata_qc_summary['duplicate_protein_uid_count']}"
)
print(
    f"Fully empty column count: {metadata_qc_summary['fully_empty_column_count']}"
)
print(
    "Normalization collision count: "
    f"{metadata_qc_summary['normalization_collision_count']}"
)
print(
    "First 10 fully empty columns: "
    f"{metadata_qc_report_payload['fully_empty_columns'][:10]}"
)

schema_drift_summary = metadata_qc_report_payload.get("schema_drift")
if schema_drift_summary is None:
    print("Schema drift diff not generated because no reference manifest was configured.")
else:
    print(f"Added columns vs reference: {schema_drift_summary['added_columns'][:10]}")
    print(f"Removed columns vs reference: {schema_drift_summary['removed_columns'][:10]}")